# `route_parser.py` — Shared VRP Utilities Guide
**Day 3 | `src/route_parser.py`**  
**Purpose:** Single source of truth for all OR-Tools VRP constants, node construction, solution parsing, and cost computation — used by both `forward_vrp.py` and `reverse_vrp.py`.

---

This notebook explains every public function in `route_parser.py`, why each design decision was made, and how to use each function interactively.

**Depends on:** `data/master_df_v2.parquet` · `data/dark_stores_final.csv`  
**Works with:** `src/forward_vrp.py` · `src/reverse_vrp.py` · `src/joint_optimizer.py`

---
## 1. Constants — Single Source of Truth

All VRP parameters live in `route_parser.py`. **Never hardcode these values in other modules.**

| Constant | Value | Why this value |
|----------|-------|----------------|
| `VEHICLE_CAPACITY_G` | 500,000 g | 500 kg — standard SP urban delivery van (Fiorino/HR) |
| `VEHICLE_SPEED_KMH` | 40 | SP metro average including traffic (~40 km/h city average) |
| `FIXED_COST_PER_ROUTE` | R$50 | Vehicle dispatch: fuel + driver base pay per route |
| `VAR_COST_PER_KM` | R$1.50 | Running cost per km (fuel + vehicle wear) |
| `SERVICE_TIME_MIN` | 5 | Minutes per stop: ring bell, hand parcel, get signature |
| `SOLVER_TIME_LIMIT_S` | 30 | GLS search budget per zone — adequate for 75-node instances |
| `MAX_CUSTOMERS_PER_ZONE` | 75 | Tractability cap — OR-Tools CVRPTW scales ~O(n²) |
| `NUM_VEHICLES` | 10 | Max vehicles available per dark store per day |
| `RETURN_PROB_THRESHOLD` | 0.30 | Flag customer for pickup if XGBoost `return_prob > 0.30` |

### Routing cost formula

$$\text{cost}(v, d) = \underbrace{50 \times v}_{\text{fixed}} + \underbrace{1.5 \times d}_{\text{variable}}$$

where $v$ = vehicles dispatched, $d$ = total km driven.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.route_parser import (
    VEHICLE_CAPACITY_G, VEHICLE_SPEED_KMH, FIXED_COST_PER_ROUTE,
    VAR_COST_PER_KM, SERVICE_TIME_MIN, SOLVER_TIME_LIMIT_S,
    MAX_CUSTOMERS_PER_ZONE, NUM_VEHICLES, RETURN_PROB_THRESHOLD,
    compute_routing_cost,
)

print("=== All VRP constants ===")
print(f"VEHICLE_CAPACITY_G      : {VEHICLE_CAPACITY_G:,} g  ({VEHICLE_CAPACITY_G/1000:.0f} kg)")
print(f"VEHICLE_SPEED_KMH       : {VEHICLE_SPEED_KMH} km/h")
print(f"FIXED_COST_PER_ROUTE    : R${FIXED_COST_PER_ROUTE:.2f}")
print(f"VAR_COST_PER_KM         : R${VAR_COST_PER_KM:.2f}")
print(f"SERVICE_TIME_MIN        : {SERVICE_TIME_MIN} min/stop")
print(f"SOLVER_TIME_LIMIT_S     : {SOLVER_TIME_LIMIT_S}s per zone")
print(f"MAX_CUSTOMERS_PER_ZONE  : {MAX_CUSTOMERS_PER_ZONE}")
print(f"NUM_VEHICLES            : {NUM_VEHICLES}")
print(f"RETURN_PROB_THRESHOLD   : {RETURN_PROB_THRESHOLD}")
print()
print("=== Example routing costs ===")
for n_veh, dist_km in [(1, 50), (2, 90), (3, 120)]:
    cost = compute_routing_cost(n_veh, dist_km)
    print(f"  {n_veh} vehicle(s), {dist_km:3d} km  →  R${cost:.2f}")

---
## 2. `build_distance_matrix(coords)` — Haversine in Metres

`route_parser.build_distance_matrix()` outputs **float64 metres** for a per-zone node set (distinct from `haversine_matrix.py` which outputs km × 1000 integers for the 500-node sample).

OR-Tools time callbacks work in **minutes**, so the distance matrix is converted at solve time:
```python
speed_m_per_min = VEHICLE_SPEED_KMH * 1000 / 60    # = 666.7 m/min
time_matrix     = np.rint(dist_matrix / speed_m_per_min).astype(int)
```

### Why metres here (not km × 1000)?
For a 75-node zone, distances are 0–50 km.  
Storing in metres gives 1-metre precision as integers without overflow risk.  
`haversine_matrix.py` uses km × 1000 because it serves OR-Tools directly; `route_parser` passes float metres to `RoutingModel` via the callback (which casts to int on the fly).

In [ ]:
import numpy as np
from src.route_parser import build_distance_matrix, VEHICLE_SPEED_KMH

# 5 nodes around SP city centre
sample_coords = np.array([
    [-23.5505, -46.6333],   # Node 0 — depot (dark store, Sé)
    [-23.5489, -46.6388],   # Node 1 — customer A
    [-23.5600, -46.6250],   # Node 2 — customer B
    [-23.5350, -46.6500],   # Node 3 — customer C
    [-23.5700, -46.6100],   # Node 4 — customer D
])

dist_m = build_distance_matrix(sample_coords)

print("Distance matrix (metres, float64):")
print(np.round(dist_m, 0).astype(int))
print()

speed_m_per_min = VEHICLE_SPEED_KMH * 1000 / 60
time_matrix     = np.rint(dist_m / speed_m_per_min).astype(int)
print("Travel time matrix (minutes at 40 km/h):")
print(time_matrix)
print()
print(f"Depot → Node 1 : {dist_m[0,1]/1000:.3f} km  →  {time_matrix[0,1]} min travel")

---
## 3. `build_vrp_nodes()` — Zone Dict Structure

This function replaced the planned static `vrp_nodes.csv` file from Pranav by building VRP nodes dynamically from `master_df_v2.parquet` + `dark_stores_final.csv`.

### Zone dict keys

```python
zone = {
    "zone_id"     : int,              # matches dark_store_id
    "store_lat"   : float,
    "store_lon"   : float,
    "node_coords" : np.ndarray,       # shape (1 + n_customers, 2) — depot FIRST
    "demands"     : np.ndarray,       # shape (1 + n_customers,)  — depot demand = 0
    "time_windows": list[list[int]],  # [[open_min, close_min], ...] depot = [0, 1440]
    "node_ids"    : list[str],        # ["depot", order_id_1, order_id_2, ...]
    "n_customers" : int,
}
```

### Node 0 is ALWAYS the depot
`node_coords[0]` = dark store centroid.  
OR-Tools requires all vehicles to **start and end at node 0**.

### Time window assignment
```python
hour < 12   →  AM window [480, 720]     # 08:00 – 12:00
hour >= 12  →  PM window [720, 1080]    # 12:00 – 18:00
depot       →  [0, 1440]                # all-day
```
Minutes from midnight — OR-Tools works in integer minutes.

In [ ]:
import pandas as pd
from src.route_parser import build_vrp_nodes

master_df   = pd.read_parquet("../data/master_df_v2.parquet")
dark_stores = pd.read_csv("../data/dark_stores_final.csv")

zones = build_vrp_nodes(master_df, dark_stores)

z = zones[0]
print(f"Zone {z['zone_id']} — {z['n_customers']} customers")
print(f"  Depot (node 0)  : lat={z['store_lat']:.4f}  lon={z['store_lon']:.4f}")
print(f"  node_coords     : shape {z['node_coords'].shape}")
print(f"  demands         : depot={z['demands'][0]}g  "
      f"customers [{z['demands'][1:].min()}–{z['demands'][1:].max()}] g")
print(f"  total demand    : {z['demands'].sum()/1000:.1f} kg  "
      f"(capacity = {500:.0f} kg)")
print()
print("First 5 nodes (node_idx : time_window):")
for i in range(min(5, len(z['time_windows']))):
    tw = z['time_windows'][i]
    label = "depot" if i == 0 else f"node {i}"
    window = "all-day [0,1440]" if tw == [0,1440] else (
        "AM [480,720]" if tw[0] == 480 else "PM [720,1080]")
    print(f"  {i:2d} ({label:8s}): {tw}  → {window}")

---
## 4. OR-Tools Index vs Node Index — The #1 Bug Source

OR-Tools uses **two separate index spaces** internally. Confusing them causes silently wrong routes.

| Index type | When used | How to get it |
|-----------|-----------|---------------|
| **Node index** | Your array subscript (0=depot, 1..N=customers) | `manager.IndexToNode(routing_idx)` |
| **Routing index** | OR-Tools internal | `manager.NodeToIndex(node_idx)` |

```python
# WRONG — using OR-Tools routing index as array subscript:
def dist_cb_WRONG(i, j):
    return int(dist_matrix[i][j])                    # ← BUG

# CORRECT — always convert via IndexToNode():
def dist_cb_CORRECT(i, j):
    return int(dist_matrix[manager.IndexToNode(i)]
                          [manager.IndexToNode(j)])  # ← correct

# Same rule for demand and time callbacks:
def demand_cb(i):
    return int(demands[manager.IndexToNode(i)])

def time_cb(i, j):
    node_i = manager.IndexToNode(i)
    return int(time_matrix[node_i][manager.IndexToNode(j)]) + (
        SERVICE_TIME_MIN if node_i != 0 else 0  # no service time at depot
    )
```

---
## 5. `parse_solution()` — Reading OR-Tools Routes

After OR-Tools solves, `assignment` is an opaque object. `parse_solution()` extracts it into a clean DataFrame by traversing each vehicle route.

### Route traversal pattern
```python
for vehicle in range(n_vehicles):
    index = routing.Start(vehicle)          # first routing index for this vehicle
    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)   # your node index
        # record (node, cumulative_distance) ...
        index = assignment.Value(routing.NextVar(index))   # advance to next stop
```

### Output DataFrame columns

| Column | Type | Description |
|--------|------|-------------|
| `vehicle_id` | int | 0 .. n_vehicles-1 |
| `stop_sequence` | int | 0 = depart depot, 1..N = customer stops, last = return depot |
| `node_idx` | int | Zone-local node index (0 = depot) |
| `node_id` | str | Original order ID or "depot" |
| `lat`, `lon` | float | Coordinates of this stop |
| `cumulative_distance_km` | float | Total km driven to reach this stop |
| `zone_id` | int | Added by `solve_cvrptw()` post-parse |

In [ ]:
import json

with open("../outputs/forward_routes.json") as f:
    routes_json = json.load(f)

routes_df  = pd.read_csv("../outputs/forward_routes.csv")

print("=== forward_routes.csv columns ===")
print(list(routes_df.columns))
print()
print("=== Zone 0, Vehicle 0 — full stop sequence ===")
z0v0 = routes_df[(routes_df["zone_id"]==0) & (routes_df["vehicle_id"]==0)]
print(z0v0[["stop_sequence","node_idx","node_id","lat","lon",
            "cumulative_distance_km"]].to_string(index=False))
print()
print(f"All zones in routes_df     : {routes_df['zone_id'].nunique()}")
print(f"Total stops (all zones)    : {len(routes_df)}")
print(f"Customer stops (node_idx≠0): {len(routes_df[routes_df['node_idx']!=0])}")

---
## 6. `save_routes()` — Output File Format Reference

### `forward_routes.json` structure
```json
{
  "zone_0": {
    "zone_id": 0,
    "solved": true,
    "total_dist_km": 94.0,
    "n_vehicles": 2,
    "routing_cost_R$": 241.0,
    "routes": [
      {
        "vehicle_id": 0,
        "stops": [
          {"stop": 0, "node_idx": 0, "node_id": "depot",  "lat": -23.55, "lon": -46.63, "cum_dist_km": 0.0},
          {"stop": 1, "node_idx": 12, "node_id": "abc123", "lat": -23.52, "lon": -46.63, "cum_dist_km": 2.1}
        ]
      }
    ]
  }
}
```

### Important: `node_idx` is zone-local
`node_idx` is the index **within that zone's node list** — not the global `vrp_nodes.csv` row number.  
`node_idx = 0` → depot. `node_idx = k` (k ≥ 1) → k-th customer sampled for this zone.

In [ ]:
# Demonstrate JSON structure and node_idx interpretation
zone_key  = list(routes_json.keys())[0]
zone_data = routes_json[zone_key]

print(f"JSON key : '{zone_key}'")
print(f"Fields   : {list(zone_data.keys())}")
print(f"Solved   : {zone_data['solved']}")
print(f"Zones    : {len(routes_json)} total")
print()

first_route = zone_data["routes"][0]
print(f"Vehicle {first_route['vehicle_id']} — first 4 stops:")
for s in first_route["stops"][:4]:
    label = "DEPOT" if s["node_idx"] == 0 else f"customer node_idx={s['node_idx']}"
    print(f"  stop {s['stop']:2d} [{label}]: "
          f"lat={s['lat']:.4f}  lon={s['lon']:.4f}  "
          f"cum_dist={s['cum_dist_km']:.2f} km")